<a href="https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1**

Pages with stronger engagement signals tend to perform better in search.

**Methodology question**

I would ask how the outcome label was created and whether engagement variables were collected before or after the outcome window. If the outcome and the features overlap in time, the reported performance could be inflated because of leakage.

**Finding 2**

Historical content performance can help identify optimization opportunities.

**Methodology question**

I would ask whether validation was performed using grouped or time-aware splits. If pages from the same client appear in both the training and testing sets, the reported results might be overly optimistic.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

For Week 5, I used a random train-test split. For Week 6, I repeated the experiment using a grouped split to reduce the risk of information leakage.

The goal is not to maximize the score but to obtain a more realistic estimate of performance.

The target variable is defined as:

- impressions_90d > 1000
- ctr < 0.5

This produces pages that may be underperforming despite receiving substantial visibility.

In [13]:
!git clone https://github.com/ch-jsk/jsk-flyrank-ml-internship.git

Cloning into 'jsk-flyrank-ml-internship'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 152 (delta 61), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 1.87 MiB | 15.59 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [14]:
%cd jsk-flyrank-ml-internship

/content/jsk-flyrank-ml-internship/jsk-flyrank-ml-internship


In [15]:
!ls data/raw

content_refresh_anonymized.csv


In [16]:
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

In [17]:
#create a target
df["target"] = (
    df["ctr"] < 0.5
).astype(int)

df["target"].value_counts()

,count
target,
1,25750
0,4250


In [18]:
#select feature
features = [
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "engagement_rate",
    "search_volume",
    "word_count"
]

X = df[features].fillna(0)
y = df["target"]


In [19]:
#split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ---------- target ----------

df["target"] = (
    (df["impressions_90d"] > 1000)
    & (df["ctr"] < 0.5)
).astype(int)

# ---------- features ----------

features = [
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "engagement_rate",
    "search_volume",
    "word_count"
]

df_model = df.dropna(subset=features)

X = df_model[features]
y = df_model["target"]

# ==========================================================
# BEFORE (random split)
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

before_score = accuracy_score(y_test, predictions)

# ==========================================================
# AFTER (grouped split)
# ==========================================================

groups = df_model["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model.fit(X_train, y_train)

predictions = model.predict(X_test)

after_score = accuracy_score(y_test, predictions)

print("Random split accuracy:", before_score)
print("Grouped split accuracy:", after_score)

Random split accuracy: 0.9200799200799201
Grouped split accuracy: 0.9284972299168975


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I deliberately checked whether any label-derived features were present.

The following columns were excluded:

- trend_direction
- trend_pct
- is_declining_label

These variables either directly define the target or contain information that would not be available at the decision point.

In [21]:
leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

for column in leakage_columns:
    print(column, column in df.columns)

trend_direction True
trend_pct True
is_declining_label False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim:

"The model identifies pages that need optimization."

Rewritten claim:

"The model identifies pages whose observed characteristics are associated with lower CTR despite relatively high visibility. These results should be interpreted as decision-support signals rather than proof of causation."

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.